## Cell 1: Setup
**What this demonstrates**: IRCoT needs two model tiers -- Haiku for cheap per-step reasoning and trigger detection, Sonnet for final answer synthesis. It also needs a retrieval corpus (Chroma + OpenAI embeddings) so the loop can fetch documents mid-reasoning.
**Expected output**: OK Setup complete with both model names, corpus path, and masked key suffixes.

In [ ]:
# -- Install dependencies --------------------------------------------------
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv anthropic

# -- Standard library -------------------------------------------------------
import os
import re
import time
import pathlib
from dataclasses import dataclass, field
from typing import Any

# -- Third-party ------------------------------------------------------------
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -- Load API keys ----------------------------------------------------------
load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set -- add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set -- add it to .env'

# -- Constants --------------------------------------------------------------
# Haiku: per-step reasoning + trigger detection (many calls, keep cost low)
STEP_MODEL    = 'claude-haiku-4-5-20251001'
# Sonnet: final answer synthesis (one call, reasoning quality matters)
ANSWER_MODEL  = 'claude-sonnet-4-6'
EMBED_MODEL   = 'text-embedding-3-small'
CHROMA_DIR    = './chroma_ircot'
MAX_STEPS     = 4   # hard ceiling on reasoning steps
MAX_CTX_CHUNKS = 6  # max chunks to accumulate across all retrieval calls
CHUNK_SIZE    = 400
TOP_K         = 3   # chunks retrieved per step

# -- Client initialisation --------------------------------------------------
client:     Anthropic        = Anthropic()
embeddings: OpenAIEmbeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('OK Setup complete')
print(f'  Step model:    {STEP_MODEL}  (reasoning steps + trigger detection)')
print(f'  Answer model:  {ANSWER_MODEL}  (final synthesis)')
print(f'  Embed model:   {EMBED_MODEL}')
print(f'  Max steps:     {MAX_STEPS}')
print(f'  Max ctx chunks:{MAX_CTX_CHUNKS}')
print(f'  Anthropic key: ...{_anthropic_key[-4:]}')
print(f'  OpenAI key:    ...{_openai_key[-4:]}')

## Cell 2: Data -- Basel III + Lending Policy Corpus
**What this demonstrates**: Build the retrieval corpus that the IRCoT loop will query mid-reasoning. Both documents are chunked and indexed in Chroma. Basel III covers capital requirements and risk weights; the lending policy covers loan approval criteria, DTI thresholds, and default procedures. Together they provide the two knowledge sources the reasoning loop will need to interleave.
**Expected output**: Chunk counts per document, Chroma vector count, and a preview of the types of content each document contributes.

In [ ]:
# -- Locate sample data ----------------------------------------------------
_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data/'

DOCS: dict[str, str] = {
    'Basel III Capital Requirements': (BASE_DIR / 'basel_iii_excerpt.txt').read_text(encoding='utf-8'),
    'Meridian Bank Lending Policy':   (BASE_DIR / 'fintech_policy.txt').read_text(encoding='utf-8'),
}

# -- Chunk and index --------------------------------------------------------
splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '\n', '. ', ' '],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=60,
)

all_chunks: list[str] = []
chunk_meta: list[dict] = []

for doc_name, doc_text in DOCS.items():
    for i, chunk in enumerate(splitter.split_text(doc_text)):
        all_chunks.append(chunk)
        chunk_meta.append({'source': doc_name, 'chunk_idx': len(all_chunks) - 1})

lc_docs = [
    Document(page_content=chunk, metadata={**chunk_meta[i], 'global_idx': i})
    for i, chunk in enumerate(all_chunks)
]
vectorstore = Chroma.from_documents(
    documents=lc_docs,
    embedding=embeddings,
    collection_name='ircot_corpus',
    persist_directory=CHROMA_DIR,
    collection_metadata={'hnsw:space': 'cosine'},
)

print(f'Corpus: {len(DOCS)} documents  ->  {len(all_chunks)} chunks')
for doc_name in DOCS:
    n = sum(1 for m in chunk_meta if m['source'] == doc_name)
    print(f'  {doc_name:42s}  {n} chunks')
print(f'Chroma: {vectorstore._collection.count()} vectors')
print()
print('What each document contributes to the IRCoT loop:')
print('  Basel III  -> risk weight tables, capital ratio minimums, RWA calculations')
print('  Lending policy -> default procedures, DTI thresholds, FICO tiers, HELOC rules')
print()
print('The IRCoT loop will retrieve from this shared corpus mid-reasoning.')
print('Each retrieval call uses a search query formulated from the latest reasoning step.')

## Cell 3: Core -- IRCoT Loop
**What this demonstrates**: Four functions compose the IRCoT pipeline. `generate_reasoning_step` produces one CoT sentence at a time. `detect_and_formulate` classifies whether that step needs a retrieval and extracts the search query if so. `ircot_retrieve` runs the retrieval. `ircot_loop` drives the interleaved cycle up to `MAX_STEPS`, accumulating context and building the reasoning trace. `synthesize_answer` produces the final grounded answer from the complete trace.
**Expected output**: All five functions defined; a smoke-test showing the trigger detector classifying two example reasoning steps.

In [ ]:
# -- Dataclass for a single reasoning step ---------------------------------
@dataclass
class ReasoningStep:
    step_num:       int
    text:           str            # the CoT sentence
    action:         str            # 'retrieve' | 'continue' | 'done'
    search_query:   str | None     # set when action == 'retrieve'
    chunks_retrieved: list[dict] = field(default_factory=list)


# ==========================================================================
# 1. Generate ONE reasoning step
# ==========================================================================
def generate_reasoning_step(
    query:         str,
    prior_steps:   list[str],
    context_chunks: list[dict],
) -> str:
    # Produce one sentence of chain-of-thought reasoning.
    # Prior steps and accumulated retrieved context are included so each
    # new step can build on what has already been reasoned and retrieved.
    context_text = ''
    if context_chunks:
        context_text = 'Retrieved context so far:\n' + '\n---\n'.join(
            f"[{c['source']}] {c['text'][:300]}"
            for c in context_chunks
        )
    prior_text = '\n'.join(f'Step {i+1}: {s}' for i, s in enumerate(prior_steps))

    prompt = (
        f'Question: {query}\n\n'
        + (f'{context_text}\n\n' if context_text else '')
        + (f'Reasoning so far:\n{prior_text}\n\n' if prior_text else '')
        + 'Write the next single reasoning step as one sentence. '
        + 'If you need a specific piece of information to continue, say '
        + '"I need to look up [topic]" or "I need to check [source] for [fact]". '
        + 'If the reasoning is complete and you can state the answer, '
        + 'begin your sentence with "ANSWER:". '
        + 'Write only one sentence.'
    )
    response = client.messages.create(
        model=STEP_MODEL,
        max_tokens=120,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response.content[0].text.strip()


# ==========================================================================
# 2. Detect retrieval need and formulate the search query
# ==========================================================================
def detect_and_formulate(step: str, query: str) -> tuple[str, str | None]:
    # Classify a reasoning step into one of three actions:
    #   'retrieve' -- step identifies a knowledge gap; return a search query
    #   'continue' -- step is an inference from known information; no retrieval needed
    #   'done'     -- step begins with ANSWER: or signals the chain is complete
    #
    # Fast path: check for explicit termination signal.
    if step.upper().startswith('ANSWER:') or step.upper().startswith('FINAL ANSWER'):
        return 'done', None

    # Fast path: check for strong linguistic retrieval signals.
    # These phrases are reliable indicators that the step has identified a gap.
    retrieval_signals = [
        'i need to look up', 'i need to check', 'i need to find',
        'i need to retrieve', 'i need to verify', 'i should look up',
        'need to consult', 'need to reference', 'need to know',
        'i need to determine', "let me check", "let me look",
    ]
    step_lower = step.lower()
    if any(sig in step_lower for sig in retrieval_signals):
        # Extract the search topic from the step text using a targeted LLM call.
        # We ask Haiku to restate the gap as a short keyword search query.
        extract_prompt = (
            f'Reasoning step: "{step}"\n\n'
            f'Original question: "{query}"\n\n'
            'Restate the information need from this step as a short '
            'keyword search query (5-10 words). Return ONLY the query text.'
        )
        resp = client.messages.create(
            model=STEP_MODEL,
            max_tokens=40,
            messages=[{'role': 'user', 'content': extract_prompt}],
        )
        search_query = resp.content[0].text.strip().strip('"').strip("'")
        return 'retrieve', search_query

    # Slow path: ask the LLM to classify steps with no obvious linguistic signal.
    # This catches implicit knowledge gaps that don't use retrieval-signal phrases.
    classify_prompt = (
        f'Reasoning step: "{step}"\n\n'
        'Does this step assert a specific fact it cannot infer from the original '
        'question alone -- something that would require looking up a policy, '
        'regulation, or document?\n'
        'Answer with exactly one of:\n'
        'RETRIEVE: <short search query>\n'
        'CONTINUE\n'
    )
    resp = client.messages.create(
        model=STEP_MODEL,
        max_tokens=40,
        messages=[{'role': 'user', 'content': classify_prompt}],
    )
    classification = resp.content[0].text.strip()
    if classification.upper().startswith('RETRIEVE:'):
        search_query = classification[len('RETRIEVE:'):].strip()
        return 'retrieve', search_query
    return 'continue', None


# ==========================================================================
# 3. Retrieval call
# ==========================================================================
def ircot_retrieve(search_query: str, k: int = TOP_K) -> list[dict]:
    # Dense cosine similarity retrieval against the shared corpus.
    # Returns plain dicts (not LangChain Documents) for easy accumulation.
    raw = vectorstore.similarity_search_with_score(search_query, k=k)
    return [
        {
            'text':   doc.page_content,
            'source': doc.metadata['source'],
            'score':  round(1.0 - dist / 2.0, 4),  # cosine similarity in [0,1]
        }
        for doc, dist in raw
    ]


# ==========================================================================
# 4. IRCoT loop
# ==========================================================================
def ircot_loop(
    query:      str,
    max_steps:  int = MAX_STEPS,
) -> dict[str, Any]:
    # Interleave reasoning and retrieval until done or max_steps reached.
    #
    # Each iteration:
    #   1. Generate one CoT sentence from current state
    #   2. Classify: retrieve / continue / done
    #   3. If retrieve: fetch chunks, add to accumulated context (deduplicating)
    #   4. If done: break
    #   5. Enforce MAX_CTX_CHUNKS to prevent context overflow
    reasoning_steps:  list[ReasoningStep] = []
    context_chunks:   list[dict]           = []  # accumulated across all retrievals
    seen_chunk_texts: set[str]              = set()  # deduplication key
    step_texts:       list[str]             = []  # plain text for the prompt
    total_llm_calls  = 0
    total_ret_calls  = 0

    for step_num in range(1, max_steps + 1):
        # --- Generate reasoning step ---
        step_text = generate_reasoning_step(query, step_texts, context_chunks)
        total_llm_calls += 1

        # --- Classify: retrieve / continue / done ---
        action, search_query = detect_and_formulate(step_text, query)
        total_llm_calls += 1  # detect_and_formulate always calls the LLM

        step = ReasoningStep(
            step_num=step_num,
            text=step_text,
            action=action,
            search_query=search_query,
        )

        if action == 'retrieve' and search_query:
            # --- Retrieve and accumulate context ---
            retrieved = ircot_retrieve(search_query)
            total_ret_calls += 1
            # Deduplicate: skip chunks already in the accumulated context
            new_chunks = [
                c for c in retrieved
                if c['text'] not in seen_chunk_texts
            ]
            for c in new_chunks:
                seen_chunk_texts.add(c['text'])
            step.chunks_retrieved = new_chunks
            # Enforce context cap -- evict oldest if over limit
            context_chunks = (context_chunks + new_chunks)[-MAX_CTX_CHUNKS:]

        reasoning_steps.append(step)
        step_texts.append(step_text)

        if action == 'done':
            break

    return {
        'query':          query,
        'reasoning_steps': reasoning_steps,
        'context_chunks': context_chunks,
        'total_llm_calls': total_llm_calls,
        'total_ret_calls': total_ret_calls,
        'steps_taken':    len(reasoning_steps),
        'hit_max_steps':  len(reasoning_steps) == max_steps and reasoning_steps[-1].action != 'done',
    }


# ==========================================================================
# 5. Final answer synthesis
# ==========================================================================
def synthesize_answer(loop_result: dict[str, Any], system: str) -> str:
    # Produce the final grounded answer from the complete reasoning trace
    # and all accumulated retrieved context.
    # Uses Sonnet for this single high-quality call.
    trace = '\n'.join(
        f'Step {s.step_num}: {s.text}'
        for s in loop_result['reasoning_steps']
    )
    context = '\n---\n'.join(
        f"[{c['source']}] {c['text']}"
        for c in loop_result['context_chunks']
    )
    user_msg = (
        f"Question: {loop_result['query']}\n\n"
        f"Reasoning trace:\n{trace}\n\n"
        f"Retrieved context:\n{context}"
    )
    resp = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=600,
        system=system,
        messages=[{'role': 'user', 'content': user_msg}],
    )
    return resp.content[0].text


# -- Smoke test: trigger detector on two example steps ----------------------
print('Smoke test -- trigger detector:')
test_cases = [
    'A loan default triggers credit loss recognition on the bank balance sheet.',
    'I need to look up the risk weight for defaulted retail exposures under Basel III.',
    'ANSWER: The capital impact is calculated as RWA multiplied by the CET1 ratio.',
]
for step in test_cases:
    action, sq = detect_and_formulate(step, 'capital implications of loan default')
    sq_str = f'  search: {sq!r}' if sq else ''
    print(f'  [{action:8s}]{sq_str}')
    print(f'           step: {step[:70]!r}')
    print()

## Cell 4: Run -- Basel III Capital Implications of Loan Default
**What this demonstrates**: The full IRCoT loop on a multi-step query that requires both Basel III capital rules and lending policy default procedures. Watch the reasoning trace build step by step, with retrieval firing when the model identifies a specific knowledge gap it cannot resolve from prior context.
**Expected output**: Step-by-step reasoning trace with retrieval triggers annotated, accumulated context chunks, final grounded answer, and full latency and cost breakdown.

In [ ]:
QUERY = (
    'If a borrower defaults on a $500K loan, what are the capital implications '
    'under Basel III and what recovery procedures does the lending policy specify?'
)

SYSTEM = (
    'You are a bank risk analyst. '
    'Answer using ONLY the provided reasoning trace and retrieved context. '
    'For every claim, cite the source document and specific section or metric. '
    'Show the capital calculation explicitly: RWA = EAD x Risk Weight, '
    'Capital = RWA x minimum capital ratio.'
)

print(f'Query: {QUERY!r}')
print(f'Max steps: {MAX_STEPS}  |  Max context chunks: {MAX_CTX_CHUNKS}')
print()
print('Running IRCoT loop...')
print()

t_total = time.perf_counter()
loop_result = ircot_loop(QUERY)
loop_ms = (time.perf_counter() - t_total) * 1000

# -- Print reasoning trace as it built ------------------------------------
for step in loop_result['reasoning_steps']:
    action_badge = {'retrieve': '[RETRIEVE]', 'continue': '[CONTINUE]', 'done': '[DONE   ]'}.get(step.action, '[?]')
    print(f'Step {step.step_num} {action_badge}')
    print(f'  Reasoning: {step.text}')
    if step.search_query:
        print(f'  Search:    {step.search_query!r}')
    if step.chunks_retrieved:
        for c in step.chunks_retrieved:
            src     = c['source'][:38]
            preview = c['text'][:80].replace('\n', ' ')
            print(f'  Retrieved: [{src}] sim={c["score"]:.3f}  {preview!r}...')
    print()

# -- Synthesise final answer -----------------------------------------------
print('Synthesising final answer...')
t_synth = time.perf_counter()
answer: str = synthesize_answer(loop_result, SYSTEM)
synth_ms = (time.perf_counter() - t_synth) * 1000

print()
print('=' * 65)
print('FINAL ANSWER')
print('=' * 65)
print(answer)
print('=' * 65)
print()
print(f'Steps taken:    {loop_result["steps_taken"]}  (max: {MAX_STEPS})')
print(f'LLM calls:      {loop_result["total_llm_calls"]}  (step gen + trigger detection)')
print(f'Retrieval calls:{loop_result["total_ret_calls"]}')
print(f'Context chunks: {len(loop_result["context_chunks"])}  accumulated')
print(f'Loop time:      {loop_ms:.0f} ms')
print(f'Synthesis time: {synth_ms:.0f} ms')
print(f'Total time:     {loop_ms + synth_ms:.0f} ms')
if loop_result['hit_max_steps']:
    print('WARNING: hit MAX_STEPS without reaching done -- consider increasing MAX_STEPS')

## Cell 5: Inspect -- Reasoning Chain, Retrieval Triggers, and Context
**What this demonstrates**: Visualise the full IRCoT execution: which steps triggered retrieval and why, what was retrieved at each step, how the accumulated context grew, and a comparison against naive single-shot RAG to show what interleaved retrieval adds.
**Expected output**: Per-step breakdown table, context accumulation timeline, source attribution summary, and a single-shot RAG baseline for comparison.

In [ ]:
# -- Per-step breakdown table ----------------------------------------------
print('IRCoT step-by-step breakdown:')
print(f'  {"Step":<5}  {"Action":<10}  {"Search query":<36}  {"Chunks":<6}  Reasoning (truncated)')
print('  ' + '-' * 100)
for s in loop_result['reasoning_steps']:
    sq      = (s.search_query or '')[:34]
    n_chunks = len(s.chunks_retrieved)
    text    = s.text[:45].replace('\n', ' ')
    print(f'  {s.step_num:<5}  {s.action:<10}  {sq:<36}  {n_chunks:<6}  {text!r}...')
print()

# -- Context accumulation timeline -----------------------------------------
print('Context accumulation across steps:')
running_total = 0
sources_seen: list[str] = []
for s in loop_result['reasoning_steps']:
    new = len(s.chunks_retrieved)
    running_total += new
    for c in s.chunks_retrieved:
        src = c['source']
        if src not in sources_seen:
            sources_seen.append(src)
    bar = '#' * new
    print(f'  Step {s.step_num}: +{new} chunks  (total: {running_total})  [{bar}]')
    if s.chunks_retrieved:
        for c in s.chunks_retrieved:
            print(f'    <- [{c["source"][:38]}] sim={c["score"]:.3f}')
print()

# -- Source attribution summary --------------------------------------------
print('Source attribution in final context:')
source_counts: dict[str, int] = {}
for c in loop_result['context_chunks']:
    source_counts[c['source']] = source_counts.get(c['source'], 0) + 1
for src, count in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f'  {count} chunk(s)  <-  {src}')
print()

# -- Single-shot RAG baseline comparison -----------------------------------
# Run one retrieval with the original query -- no reasoning loop.
# Compare what it surfaces against what IRCoT accumulated.
print('Baseline comparison: single-shot RAG (one retrieval, original query):')
baseline_chunks = ircot_retrieve(QUERY, k=TOP_K)
baseline_sources = {c['source'] for c in baseline_chunks}
ircot_sources    = {c['source'] for c in loop_result['context_chunks']}

print(f'  Single-shot retrieved: {len(baseline_chunks)} chunks')
for c in baseline_chunks:
    preview = c['text'][:70].replace('\n', ' ')
    print(f'    [{c["source"][:38]}] sim={c["score"]:.3f}  {preview!r}...')
print()
print(f'  IRCoT accumulated:     {len(loop_result["context_chunks"])} chunks across {loop_result["total_ret_calls"]} targeted retrieval(s)')
print()

sources_only_ircot    = ircot_sources - baseline_sources
sources_only_baseline = baseline_sources - ircot_sources
if sources_only_ircot:
    print(f'  Sources found by IRCoT but NOT by single-shot: {sources_only_ircot}')
    print('  -> IRCoT retrieved from a source the original query did not surface.')
if sources_only_baseline:
    print(f'  Sources found by single-shot but NOT by IRCoT: {sources_only_baseline}')
if not sources_only_ircot and not sources_only_baseline:
    print('  Same source documents surfaced -- IRCoT benefit is in targeted chunk selection.')
print()
print('Key difference: IRCoT used', loop_result['total_ret_calls'],
      'targeted retrieval(s) with queries derived from reasoning;')
print('single-shot used 1 retrieval with the original (often too broad) query.')

## Cell 6: Fintech -- Multi-Step HELOC Eligibility Assessment
**What this demonstrates**: IRCoT applied to a complex compliance calculation that requires sequential lookups across multiple sections of the lending policy. The reasoning chain must verify FICO eligibility, calculate DTI, check product parameters, and confirm regulatory alignment -- four distinct sub-questions each requiring a different policy section, none of which can be pre-fetched without first completing the prior reasoning step.
**Expected output**: Full reasoning trace with 3-4 targeted retrievals, a structured eligibility determination with citations, and a summary of why IRCoT is the right pattern for this class of sequential decision query.

In [ ]:
# -- Multi-step HELOC eligibility query ------------------------------------
# Scenario: a loan officer must determine whether an applicant qualifies
# for a Home Equity Line of Credit (HELOC) under the bank's lending policy.
# The determination requires sequential sub-checks:
#   1. Does the FICO score meet the product minimum?
#   2. Calculate DTI: monthly debt / gross monthly income
#   3. Does DTI fall within the HELOC product parameters?
#   4. Are there any Basel III capital requirements the bank must hold?
# Each sub-check requires a different section of the corpus.

HELOC_QUERY = (
    'An applicant has a 640 FICO score, $75,000 annual income, and '
    '$2,500 in monthly debt obligations. '
    'Do they qualify for a $250,000 HELOC under the Meridian Bank lending policy, '
    'and what capital must the bank hold against this exposure under Basel III?'
)

HELOC_SYSTEM = (
    'You are a loan officer assistant performing a structured eligibility determination. '
    'Answer using ONLY the provided reasoning trace and retrieved context. '
    'Structure your answer in sections: '
    '(1) FICO Assessment, (2) DTI Calculation, (3) Product Eligibility, '
    '(4) Capital Requirement. '
    'For every criterion, cite the specific policy section and threshold. '
    'State a clear final determination: APPROVED or DECLINED, with justification.'
)

print('HELOC eligibility query:')
print(f'  {HELOC_QUERY!r}')
print()

t0 = time.perf_counter()
heloc_result = ircot_loop(HELOC_QUERY, max_steps=MAX_STEPS)
loop_ms = (time.perf_counter() - t0) * 1000

# -- Print trace with retrieval annotations --------------------------------
print('Reasoning trace:')
for step in heloc_result['reasoning_steps']:
    badge = {'retrieve': 'RETRIEVE', 'continue': 'CONTINUE', 'done': 'DONE    '}.get(step.action, '?')
    print(f'  Step {step.step_num} [{badge}]: {step.text}')
    if step.search_query:
        print(f'             search: {step.search_query!r}')
    for c in step.chunks_retrieved:
        preview = c['text'][:90].replace('\n', ' ')
        print(f'             fetched: [{c["source"][:35]}]  {preview!r}...')
    print()

# -- Synthesise eligibility determination ----------------------------------
t_synth = time.perf_counter()
heloc_answer: str = synthesize_answer(heloc_result, HELOC_SYSTEM)
synth_ms = (time.perf_counter() - t_synth) * 1000

print('=' * 65)
print('HELOC ELIGIBILITY DETERMINATION')
print('=' * 65)
print(heloc_answer)
print('=' * 65)

# -- Loop statistics -------------------------------------------------------
print()
print(f'Steps: {heloc_result["steps_taken"]}  |  '
      f'Retrievals: {heloc_result["total_ret_calls"]}  |  '
      f'LLM calls: {heloc_result["total_llm_calls"]}  |  '
      f'Loop: {loop_ms:.0f} ms  |  Synthesis: {synth_ms:.0f} ms')
print()

# -- Why IRCoT is the right pattern for this query type -------------------
print('Why IRCoT fits this query class:')
print('  Sub-question 1 (FICO threshold) -- only discoverable after reading the query')
print('  Sub-question 2 (DTI calculation) -- requires Q1 result to know which tier')
print('  Sub-question 3 (HELOC parameters) -- requires DTI result to compare against limit')
print('  Sub-question 4 (capital requirement) -- independent but follows the eligibility chain')
print()
print('A single upfront retrieval for the full query would surface generic loan policy')
print('sections and likely miss the HELOC-specific parameters or Basel III risk weights.')
print('IRCoT retrieved targeted sections at each step because the reasoning identified')
print('the specific knowledge gap before issuing each retrieval call.')